# 03 · Chuẩn hóa từ train: công thức, provenance và tác động
**Câu hỏi:** mean/std được học từ đâu; chuẩn hóa thay đổi điều gì; có thể khôi phục mV để giải thích tín hiệu không?

**Đầu vào:** records và các .npy accepted. **Đầu ra:** normalization.json, gồm 12 mean/std, số nhãn dương/âm, pos_weight và dấu vết nguồn fit. **Tiếp theo:** 04 audit.

Với lead \(l\), gộp mọi điểm thời gian của mọi ECG train accepted:
\[\mu_l=\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}x_{ilt},\quad
\sigma_l=\sqrt{\frac{1}{NT}\sum_{i,t}(x_{ilt}-\mu_l)^2},\quad
z_{ilt}=\frac{x_{ilt}-\mu_l}{\sigma_l}.\]

Dùng population std (ddof=0). Validation/test dùng lại tham số train. File .npy vẫn là mV; z chỉ tạo khi đọc batch. Không cần lưu thêm toàn bộ một dataset chuẩn hóa.

Bản sửa 01/11/2026 chưa chạy toàn bộ. Các cell minh họa nhỏ không thay thế phép fit toàn bộ train.

In [ ]:
from pathlib import Path
import os
import sys

def tim_thu_muc_project():
    explicit = os.environ.get("PTBXL_PROJECT_ROOT")
    candidates = [Path(explicit).expanduser().resolve()] if explicit else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ptbxl_pipeline").is_dir() and (candidate / "requirements-ptbxl.txt").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy project. Đặt PTBXL_PROJECT_ROOT tới thư mục chứa ptbxl_pipeline và requirements-ptbxl.txt.")

ROOT = tim_thu_muc_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW_DIR = Path(os.environ.get("PTBXL_RAW_DIR", str(ROOT / "data/raw/ptbxl/1.0.3"))).expanduser().resolve()
CACHE_DIR = ROOT / "data/cache"
OUTPUT_DIR = Path(os.environ.get("PTBXL_OUTPUT_DIR", str(ROOT / "data/processed/ptbxl_v1.0.3_500hz_minimal"))).expanduser().resolve()
REPORT_DIR = ROOT / "artifacts/ptbxl_reports" / OUTPUT_DIR.name
TARGET_FS = 500
PREPROCESSING = "minimal"
EXPECTED_RECORDS = 21799
print("Project:", ROOT)
print("Dữ liệu gốc:", RAW_DIR)
print("Bộ xuất:", OUTPUT_DIR)

In [ ]:
from ptbxl_pipeline.core import PipelineConfig

config = PipelineConfig(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    target_fs=TARGET_FS,
    preprocessing=PREPROCESSING,
    expected_records=EXPECTED_RECORDS,
)

## 1. Ví dụ số nhỏ để thấy vì sao không fit trên validation
Đây là ví dụ toán học nhân tạo, được ghi rõ, không phải kết quả PTB-XL. Một lead, hai ECG train, mỗi ECG ba điểm. Thêm một “validation” có giá trị lớn sẽ làm lệch mean/std nếu gộp sai.

Welford/ghép thống kê block cho phép core tính mean và tổng bình phương độ lệch mà không giữ mọi ECG trong RAM. Kết quả cần tương đương công thức trực tiếp, trong sai số số học.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

toy_train = np.array([[1., 2., 3.], [4., 5., 6.]])
toy_validation = np.array([[100., 101., 102.]])
toy_mean, toy_std = toy_train.mean(), toy_train.std(ddof=0)
count, running_mean, m2 = 0, 0.0, 0.0
for block in toy_train:
    block_count, block_mean = block.size, block.mean()
    block_m2 = np.sum((block - block_mean) ** 2)
    delta = block_mean - running_mean
    total = count + block_count
    running_mean += delta * block_count / total
    m2 += block_m2 + delta ** 2 * count * block_count / total
    count = total
np.testing.assert_allclose([running_mean, np.sqrt(m2 / count)], [toy_mean, toy_std])
display(pd.DataFrame({"cach_fit": ["đúng: chỉ train", "sai: gộp validation"], "mean": [toy_mean, np.concatenate([toy_train, toy_validation]).mean()], "std_ddof0": [toy_std, np.concatenate([toy_train, toy_validation]).std()]}))
print("Validation được biến đổi bằng train stats:", (toy_validation - toy_mean) / toy_std)

## 2. TOÀN BỘ TRAIN: tính và lưu tham số
Cell dưới đọc mọi waveform accepted/train và ghi normalization.json. Nó không fit trên validation/test. Runtime/code/cấu hình phải khớp release; không bỏ guard khi notebook báo mismatch.

Đây là thao tác I/O nhiều GB, dù bộ nhớ được giới hạn. Không thay bằng thống kê từ 32 ví dụ minh họa ở phần sau.

In [ ]:
from ptbxl_pipeline.core import fit_normalizer, LEADS, CLASSES

normalization = fit_normalizer(config)
means = np.asarray(normalization["mean"], dtype=np.float64)
stds = np.asarray(normalization["std"], dtype=np.float64)
assert means.shape == stds.shape == (12,)
assert np.isfinite(means).all() and np.isfinite(stds).all() and (stds > 0).all()
display(pd.DataFrame({"lead": LEADS, "mean_train_mV": means, "std_train_mV": stds}))
print("Đã lưu:", OUTPUT_DIR / "normalization.json")

## 3. Kiểm tra tham số thuộc đúng train của release
Đối chiếu fingerprint, checksum records, số ECG, số điểm mỗi lead và số nhãn dương. Những kiểm tra này bắt dùng nhầm file thống kê hoặc nhầm cohort; không tự chứng minh công thức đã chạy đúng trên mọi waveform. Audit 04 còn đối chiếu tính toàn vẹn của artifact.

In [ ]:
import json
import hashlib

records = pd.read_csv(OUTPUT_DIR / "records.csv", keep_default_na=False, low_memory=False)
identity = json.loads((OUTPUT_DIR / "pipeline.json").read_text(encoding="utf-8"))
train_rows = records.loc[records.split.eq("train") & records.status.eq("accepted")].sort_values("ecg_id")
label_columns = [f"y_{name}" for name in CLASSES]
records_hash = hashlib.sha256((OUTPUT_DIR / "records.csv").read_bytes()).hexdigest()
normalizer_hash = hashlib.sha256((OUTPUT_DIR / "normalization.json").read_bytes()).hexdigest()
assert normalization["fit_split"] == "train" and normalization["ddof"] == 0
assert normalization["fingerprint"] == identity["fingerprint"]
assert normalization["records_sha256"] == identity["records_sha256"] == records_hash
assert identity["normalization_sha256"] == normalizer_hash
assert normalization["train_records"] == len(train_rows)
assert normalization["samples_per_lead"] == len(train_rows) * TARGET_FS * 10
assert normalization["classes"] == list(CLASSES) and normalization["leads"] == list(LEADS)
assert normalization["positive_counts"] == train_rows[label_columns].sum().astype(int).tolist()
display(pd.Series({"fit_split": normalization["fit_split"], "ECG_train": len(train_rows), "diem_moi_lead": normalization["samples_per_lead"], "fingerprint": identity["fingerprint"]}).to_frame("gia_tri"))

## 4. Theo một ECG: chuẩn hóa rồi khôi phục mV
Dùng ECG accepted/train đầu tiên. Z không còn đơn vị mV; không đọc biên độ lâm sàng trực tiếp trên z. Phép ngược là x = z × std + mean. Vì loader trả float32, cho phép sai số làm tròn nhỏ; không đòi bitwise equality sau phép ngược.

In [ ]:
example = train_rows.iloc[0]
x_mV = np.load(OUTPUT_DIR / example.npy_path, allow_pickle=False)
z = ((x_mV.astype(np.float64) - means[:, None]) / stds[:, None]).astype(np.float32)
restored_mV = z.astype(np.float64) * stds[:, None] + means[:, None]
np.testing.assert_allclose(restored_mV, x_mV, rtol=2e-6, atol=1e-6)
display(pd.DataFrame({"lead": LEADS, "mean_ECG_mV": x_mV.mean(axis=1), "std_ECG_mV": x_mV.std(axis=1), "mean_ECG_z": z.mean(axis=1), "std_ECG_z": z.std(axis=1), "max_sai_so_khoi_phuc_mV": np.max(np.abs(restored_mV - x_mV), axis=1)}))

**Cách đọc:** mean/std của từng ECG không nhất thiết là 0/1. Ta dùng thống kê chung train cho mỗi lead, không z-score riêng từng bản ghi. Giữ quan hệ biên độ giữa các ECG là lý do quan trọng để phân biệt hai cách.

## 5. Phân bố trước/sau trên mẫu train cố định
Chỉ lấy tối đa 32 ECG, không fit lại. Mẫu này cũng không bắt buộc đạt mean 0/std 1. Histogram dùng lead II, lấy một điểm mỗi 5 mẫu để giảm số điểm vẽ; bảng per-lead dùng đủ điểm của 32 ECG. Cả hai đều không phải phân bố toàn train.

In [ ]:
import matplotlib.pyplot as plt

sample_rows = train_rows.sample(n=min(32, len(train_rows)), random_state=42).sort_values("ecg_id")
raw_sample = np.stack([np.load(OUTPUT_DIR / row.npy_path, allow_pickle=False) for row in sample_rows.itertuples(index=False)])
z_sample = ((raw_sample.astype(np.float64) - means[None, :, None]) / stds[None, :, None]).astype(np.float32)
display(pd.DataFrame({"lead": LEADS, "mean_mV_mau": raw_sample.mean(axis=(0, 2)), "std_mV_mau": raw_sample.std(axis=(0, 2)), "mean_z_mau": z_sample.mean(axis=(0, 2)), "std_z_mau": z_sample.std(axis=(0, 2))}))
lead_index = LEADS.index("II")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw_sample[:, lead_index, ::5].ravel(), bins=80)
axes[0].set(xlabel="mV", ylabel="Số điểm lấy mẫu", title="Trước chuẩn hóa — lead II")
axes[1].hist(z_sample[:, lead_index, ::5].ravel(), bins=80)
axes[1].set(xlabel="z (không đơn vị)", ylabel="Số điểm lấy mẫu", title="Dùng mean/std của toàn train")
fig.suptitle(f"Mẫu {len(sample_rows)}/{len(train_rows)} ECG accepted/train; không fit lại")
fig.tight_layout()
plt.show()

## 6. Pos_weight là ứng viên thí nghiệm, không phải lựa chọn tự động
Với từng nhãn, pos_weight = số âm / số dương trên accepted/train. Một ECG đa nhãn đóng góp riêng vào từng lớp. BCE thường và BCE có trọng số cần so sánh trên validation; weighting có thể thay đổi calibration và ngưỡng tối ưu.

Nếu một lớp không có dương/âm, phải nêu giới hạn học/đánh giá lớp đó; không tạo trọng số giả. Không nhân bản/xóa validation hoặc test để “cân bằng” chỉ số.

In [ ]:
positives = np.asarray(normalization["positive_counts"], dtype=int)
negatives = np.asarray(normalization["negative_counts"], dtype=int)
weights = np.asarray(normalization["pos_weight"], dtype=float)
assert np.array_equal(positives + negatives, np.full(5, len(train_rows)))
assert (positives > 0).all() and np.isfinite(weights).all()
np.testing.assert_allclose(weights, negatives / positives)
display(pd.DataFrame({"nhan": CLASSES, "duong_train": positives, "am_train": negatives, "pos_weight": weights, "co_ca_duong_va_am": (positives > 0) & (negatives > 0)}))

## Quyết định, chi phí và giới hạn
Giữ chuẩn hóa train-global/per-lead, áp dụng trong loader. .npy vẫn mV để đối chiếu/XAI. Không thêm một bản dữ liệu chuẩn hóa hơn 5 GB; chi phí tính phép affine xảy ra khi đọc batch và cần đo throughput trên máy train.

Đã thiết kế kiểm tra provenance, phép ngược và minh họa phân bố; chưa có kết luận chất lượng model. Nếu đổi cohort, bộ lọc hoặc tần số, phải fit lại trên train của release mới, không tái dùng tham số cũ. Chuyển sang **04_kiem_dinh_dong_goi.ipynb** để kiểm định toàn bộ.